# Colab / Local Benchmark Runner

Single-GPU benchmark entrypoint for Google Colab and local Jupyter.

Flow:
1. Detect Colab vs local notebook
2. Mount Drive when running in Colab
3. Clone/update the repo into `/content` in Colab, or reuse the current local checkout
3. Install `bench/` dependencies
4. Build or reuse `llama-server`
5. Ensure model/mmproj artifacts
6. Run a smoke test
7. Run the requested benchmark matrix with resume support


In [ ]:
REPO_URL = "https://github.com/Chedrian07/TQ_LLAMACPP_VSION_BENCH.git"
BRANCH = "main"
LLAMA_CPP_REPO_URL = "https://github.com/Chedrian07/llama.cpp.git"
LLAMA_CPP_BRANCH = "master"
LLAMA_CPP_COMMIT = "68adefd66195c02db2791c350c88639b2722cbc0"
WORKSPACE_ROOT = "/content"
DRIVE_ROOT = "/content/drive/MyDrive/tq_vlm_bench"

# User-editable run_bench.py defaults
MODEL_ID = "qwen3_vl_2b_thinking"
MODEL_QUANT = "bf16"
# Runtime groups are supported directly by run_bench.py, e.g. ["core", "prod"].
RUNTIMES = ["core", "prod"]
# Keep benchmark selection explicit so the notebook stays easy to retarget.
BENCHMARKS = ["ai2d"]
N_SAMPLES = 500
# Explicitly keep request parallelism at 1 in Colab.
PARALLEL_OVERRIDE = 1
# Append arbitrary run_bench.py flags here, e.g. ["--seed", "123"].
EXTRA_RUN_BENCH_ARGS = []
RUN_KV_DUMP = True

RESULTS_TAG = f"colab_{MODEL_ID}_{MODEL_QUANT}"
RESUME = True
FORCE_REBUILD_SERVER = False
FORCE_REDOWNLOAD_MODEL = False


In [ ]:
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    PROFILE = "colab"
else:
    PROFILE = "local"
    print("google.colab not available; running in local notebook mode.")
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    local_repo_root = next(
        (
            path for path in candidates
            if (path / ".git").exists() and (path / "bench").exists() and (path / "README.md").exists()
        ),
        None,
    )
    if local_repo_root is None:
        raise RuntimeError("Could not locate the repo root from the current local notebook working directory.")
    if WORKSPACE_ROOT == "/content":
        WORKSPACE_ROOT = str(local_repo_root.parent)
    if DRIVE_ROOT.startswith("/content/drive/"):
        DRIVE_ROOT = str((local_repo_root / ".colab-local").resolve())

print(f"IN_COLAB={IN_COLAB}")
print(f"PROFILE={PROFILE}")
print(f"WORKSPACE_ROOT={WORKSPACE_ROOT}")
print(f"DRIVE_ROOT={DRIVE_ROOT}")


In [ ]:
import subprocess
import sys
from pathlib import Path

workspace_root = Path(WORKSPACE_ROOT).expanduser().resolve()
repo_name = Path(REPO_URL.rstrip("/")).name.removesuffix(".git")
repo_root = workspace_root / repo_name

if not IN_COLAB:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    REPO_ROOT = next(
        (
            path for path in candidates
            if (path / ".git").exists() and (path / "bench").exists() and path.name == repo_name
        ),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("Could not find the current repo checkout for local notebook mode.")
else:
    if not (repo_root / ".git").exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    REPO_ROOT = repo_root

sys.path.insert(0, str((REPO_ROOT / "bench").resolve()))

from tq_bench.colab import ensure_repo_checkout

if IN_COLAB:
    REPO_ROOT = ensure_repo_checkout(REPO_URL, BRANCH, workspace_root)
print(f"Repo root: {REPO_ROOT}")


In [ ]:
from tq_bench.colab import configure_hf_cache, install_bench_editable

install_bench_editable(REPO_ROOT)
HF_PATHS = configure_hf_cache(DRIVE_ROOT)
print(HF_PATHS)


In [ ]:
from tq_bench.colab import ensure_llama_cpp_checkout, ensure_llama_server, ensure_model_artifacts

DRIVE_ROOT = Path(DRIVE_ROOT).expanduser().resolve()
if IN_COLAB:
    llama_checkout = ensure_llama_cpp_checkout(
        REPO_ROOT,
        llama_repo_url=LLAMA_CPP_REPO_URL,
        branch=LLAMA_CPP_BRANCH,
        commit=LLAMA_CPP_COMMIT or None,
    )
else:
    llama_root = REPO_ROOT / "llama.cpp"
    if not (llama_root / ".git").exists():
        raise RuntimeError("Local notebook mode expects an existing llama.cpp git checkout at REPO_ROOT/llama.cpp")
    llama_checkout = type("LocalCheckout", (), {
        "path": llama_root,
        "repo_url": subprocess.run(["git", "config", "--get", "remote.origin.url"], cwd=str(llama_root), check=True, capture_output=True, text=True).stdout.strip(),
        "commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(llama_root), check=True, capture_output=True, text=True).stdout.strip(),
    })()
artifact = ensure_llama_server(
    REPO_ROOT,
    DRIVE_ROOT,
    force_rebuild=FORCE_REBUILD_SERVER,
)
copied = ensure_model_artifacts(
    REPO_ROOT / "bench" / "configs" / "models.yaml",
    model_id=MODEL_ID,
    quant=MODEL_QUANT,
    drive_root=DRIVE_ROOT,
    force_redownload=FORCE_REDOWNLOAD_MODEL,
)
RESULTS_RUNS_DIR = DRIVE_ROOT / "results" / "runs"
RESULTS_RUNS_DIR.mkdir(parents=True, exist_ok=True)
SLOT_SAVE_PATH = REPO_ROOT / "kvcache"
SLOT_SAVE_PATH.mkdir(parents=True, exist_ok=True)

print(f"llama.cpp checkout: {llama_checkout.path}")
print(f"llama.cpp remote:   {llama_checkout.repo_url}")
print(f"llama.cpp commit:   {llama_checkout.commit}")
print(f"llama-server: {artifact.binary_path}")
print(f"llama-kv-dump: {artifact.kv_dump_binary_path}")
print(f"CUDA arch:    sm{artifact.cuda_arch}")
print(f"Model file:   {copied[MODEL_QUANT]}")
print(f"mmproj file:  {copied.get('mmproj')}")
print(f"Results dir:  {RESULTS_RUNS_DIR}")


In [ ]:
import importlib
import os
import subprocess
import sys

sys.path.insert(0, str((REPO_ROOT / "bench").resolve()))

import tq_bench.colab as tq_colab
tq_colab = importlib.reload(tq_colab)
build_run_bench_command = tq_colab.build_run_bench_command
run_command_live = getattr(tq_colab, "run_command_live", None)
if run_command_live is None:
    def run_command_live(cmd, *, cwd=None, env=None, step=None):
        if step:
            print(f"[tq-bench] {step}: {' '.join(cmd)}", flush=True)
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd is not None else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
        finally:
            proc.stdout.close()
        returncode = proc.wait()
        if returncode != 0:
            raise RuntimeError(f"Command failed with exit code {returncode}: {' '.join(cmd)}")

smoke_benchmarks = BENCHMARKS[:1] if BENCHMARKS else ["ai2d"]
SMOKE_OUTPUT = RESULTS_RUNS_DIR / f"{RESULTS_TAG}_smoke.json"
smoke_cmd = build_run_bench_command(
    REPO_ROOT,
    num=1,
    model_id=MODEL_ID,
    model_quant=MODEL_QUANT,
    runtimes=["baseline"],
    benchmarks=smoke_benchmarks,
    profile=PROFILE,
    parallel=PARALLEL_OVERRIDE,
    results_dir=RESULTS_RUNS_DIR,
    output_path=SMOKE_OUTPUT,
    slot_save_path=SLOT_SAVE_PATH,
    server_binary_path=artifact.binary_path,
    kv_dump_binary_path=artifact.kv_dump_binary_path,
)
print(" ".join(smoke_cmd))
run_command_live(smoke_cmd, cwd=REPO_ROOT, env=os.environ.copy(), step="smoke")


In [ ]:
import importlib
import os
import subprocess
import sys

sys.path.insert(0, str((REPO_ROOT / "bench").resolve()))

import tq_bench.colab as tq_colab
tq_colab = importlib.reload(tq_colab)
build_run_bench_command = tq_colab.build_run_bench_command
run_command_live = getattr(tq_colab, "run_command_live", None)
if run_command_live is None:
    def run_command_live(cmd, *, cwd=None, env=None, step=None):
        if step:
            print(f"[tq-bench] {step}: {' '.join(cmd)}", flush=True)
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd is not None else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
        finally:
            proc.stdout.close()
        returncode = proc.wait()
        if returncode != 0:
            raise RuntimeError(f"Command failed with exit code {returncode}: {' '.join(cmd)}")

OUTPUT_PATH = RESULTS_RUNS_DIR / f"{RESULTS_TAG}.json"
resume_path = OUTPUT_PATH if RESUME and OUTPUT_PATH.exists() else None
main_cmd = build_run_bench_command(
    REPO_ROOT,
    num=N_SAMPLES,
    model_id=MODEL_ID,
    model_quant=MODEL_QUANT,
    runtimes=RUNTIMES,
    benchmarks=BENCHMARKS,
    profile=PROFILE,
    parallel=PARALLEL_OVERRIDE,
    results_dir=RESULTS_RUNS_DIR,
    output_path=OUTPUT_PATH,
    resume_path=resume_path,
    slot_save_path=SLOT_SAVE_PATH,
    server_binary_path=artifact.binary_path,
    kv_dump_binary_path=artifact.kv_dump_binary_path,
    extra_args=EXTRA_RUN_BENCH_ARGS,
)
if RUN_KV_DUMP:
    main_cmd.append("--kv-dump")
print(" ".join(main_cmd))
run_command_live(main_cmd, cwd=REPO_ROOT, env=os.environ.copy(), step="main")


In [ ]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

with open(OUTPUT_PATH) as f:
    payload = json.load(f)

rows = []
for rec in payload.get("records", []):
    rows.append(
        {
            "runtime_id": rec.get("runtime_id"),
            "benchmark_id": rec.get("benchmark_id"),
            "status": rec.get("status"),
            "score": rec.get("score"),
            "n_succeeded": rec.get("n_succeeded"),
            "n_samples": rec.get("n_samples"),
            "wall_time_seconds": rec.get("wall_time_seconds"),
        }
    )

df = pd.DataFrame(rows)
display(df.sort_values(["runtime_id", "benchmark_id"]).reset_index(drop=True))
print(json.dumps(payload.get("meta", {}), indent=2))
print(f"Results file: {OUTPUT_PATH}")

if RUN_KV_DUMP:
    kv_root = REPO_ROOT / "logs" / "kv_analysis"
    run_dirs = sorted(
        [path for path in kv_root.glob(f"{MODEL_ID}_*") if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    if not run_dirs:
        print(f"No KV analysis runs found under {kv_root}")
    else:
        latest_run = run_dirs[-1]
        print(f"KV analysis root: {latest_run}")
        dump_dirs = sorted(path for path in (latest_run / "dumps").glob("*") if path.is_dir())
        print("KV dump dirs:", [path.name for path in dump_dirs])

        analysis_dirs = sorted(path for path in (latest_run / "analysis").glob("*") if path.is_dir())
        if not analysis_dirs:
            print("No comparative KV analysis directories found yet.")
        for analysis_dir in analysis_dirs:
            print(f"\n=== KV analysis: {analysis_dir.name} ===")
            report_path = analysis_dir / "report.md"
            if report_path.exists():
                display(Markdown(report_path.read_text(encoding="utf-8")))

            csv_paths = sorted(analysis_dir.glob("*.csv"))
            for csv_path in csv_paths:
                print(f"Artifact CSV: {csv_path.name}")
                display(pd.read_csv(csv_path).head())

            plot_dir = analysis_dir / "plots"
            if plot_dir.exists():
                for plot_path in sorted(plot_dir.glob("*.png")):
                    print(f"Plot: {plot_path.name}")
                    display(Image(filename=str(plot_path)))
